# RL-NDVI RecurrentPPO
Versión en notebook que entrena un `RecurrentPPO` (LSTM) sobre series NDVI.

Este cuaderno define el entorno `MultiSeriesSamplerEnv`, carga las series desde CSVs en la carpeta `SERIES_NUEVAS`, y entrena un modelo `RecurrentPPO` de `sb3_contrib`.

In [ ]:
# Instalar dependencias (ejecutar una vez en el entorno)
# !pip install stable-baselines3 sb3-contrib gym numpy pandas --quiet

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import gymnasium as gym

from gymnasium import spaces

from sb3_contrib import RecurrentPPO
from stable_baselines3.common.vec_env import DummyVecEnv

print('Imports OK')

Imports OK


In [2]:
class MultiSeriesSamplerEnv(gym.Env):
    """Wrapper que muestrea una serie por episodio para entrenar con todas las series."""

    def __init__(self, series_pairs, window=23):
        super().__init__()
        self.series_pairs = series_pairs
        self.n_series = len(series_pairs)
        self.window = window
        # length basada en la longitud mínima de las series NDVI
        self.length = min(len(s[0]) for s in series_pairs)
        # observation: ventana x 2 (NDVI, class)
        self.observation_space = spaces.Box(low=-10.0, high=10.0, shape=(window, 2), dtype=np.float32)
        self.action_space = spaces.Box(low=-10.0, high=10.0, shape=(2,), dtype=np.float32)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # elegir una serie al azar para el episodio
        self.current_series = np.random.randint(0, self.n_series)
        self.current_step = self.window
        ndvi = np.array(self.series_pairs[self.current_series][0], dtype=np.float32)[: self.length]
        cat = np.array(self.series_pairs[self.current_series][1], dtype=np.float32)[: self.length]
        self.data = np.stack([ndvi, cat], axis=1)  # shape (length, 2)
        obs = self.data[self.current_step - self.window : self.current_step].copy()
        return obs, {}

    def step(self, action):
        target = self.data[self.current_step]
        reward = -np.mean((action - target) ** 2)
        self.current_step += 1
        terminated = self.current_step >= self.length
        truncated = False
        if not terminated:
            next_obs = self.data[self.current_step - self.window : self.current_step].copy()
        else:
            next_obs = np.zeros((self.window, 2), dtype=np.float32)
        info = {}
        return next_obs, float(reward), terminated, truncated, info

In [3]:
def load_series_from_folder(folder_path):
    """Carga pares (ndvi_series, class_series) desde CSVs en la carpeta dada.
    Se asume que cada CSV tiene al menos una columna 'ndvi' y opcionalmente 'class' (0/1).
    Si no hay 'class', se llenará con ceros.
    """
    series = []
    patterns = [os.path.join(folder_path, "*_train.csv"), os.path.join(folder_path, "*_val.csv")]
    files = []
    for p in patterns:
        files.extend(glob.glob(p))
    files = sorted(list(set(files)))
    if len(files) == 0 and os.path.exists(folder_path) and folder_path.endswith('.csv'):
        files = [folder_path]

    for fp in files:
        df = pd.read_csv(fp)
        if 'ndvi' not in df.columns:
            ndvi = df.iloc[:, 0].astype(np.float32).values
        else:
            ndvi = df['ndvi'].astype(np.float32).values
        if 'class' in df.columns:
            cat = df['class'].astype(np.float32).values
        else:
            cat = np.zeros_like(ndvi, dtype=np.float32)
        series.append((ndvi, cat))
    return series

def make_env(series_pairs, window=23):
    """Crea un env vectorizado compatible con stable-baselines3 y RecurrentPPO."""
    def _init():
        env = MultiSeriesSamplerEnv(series_pairs, window=window)
        return env

    # Usar DummyVecEnv sin wrappers adicionales
    # RecurrentPPO puede manejar observaciones multidimensionales
    env = DummyVecEnv([_init])
    return env

In [4]:
# Prueba rápida: cargar algunas series y comprobar observación
data_folder = 'SERIES_NUEVAS'  # Cambia si tu carpeta es otra
series = load_series_from_folder(data_folder)
print(f'Series cargadas: {len(series)}')
if len(series) > 0:
    env = make_env(series, window=23)
    obs, info = env.reset()
    print('Observación (shape):', obs.shape)

Series cargadas: 24


ValueError: not enough values to unpack (expected 2, got 1)

In [ ]:
# Verificación del entorno antes de entrenar
print("Verificando shapes del entorno...")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")

# Entrenamiento: ajustar parámetros aquí
window = 23
timesteps = 50000  # reducir para prueba rápida
n_steps = 64  # Reducido para series más cortas
lr = 3e-4
batch_size = 32
tensorboard_log = 'logs_recurrentppo'
save_path = 'recurrentppo_ndvi_model'

# Asegurarse de que batch_size < n_steps
if batch_size > n_steps:
    batch_size = n_steps // 2
    print(f"Aajustado batch_size a {batch_size} (debe ser < n_steps)")

print(f"n_steps: {n_steps}, batch_size: {batch_size}")

# (Re)crear el entorno si no existe
if 'env' not in globals() or env.num_envs == 0:
    series = load_series_from_folder(data_folder)
    env = make_env(series, window=window)

model = RecurrentPPO(
    policy='MlpLstmPolicy',
    env=env,
    verbose=1,
    tensorboard_log=tensorboard_log,
    n_steps=n_steps,
    learning_rate=lr,
    batch_size=batch_size,
)

print("Iniciando entrenamiento...")
model.learn(total_timesteps=timesteps)
model.save(save_path)
print(f'Modelo guardado en: {save_path}')

**Notas y siguientes pasos**:
- Ajusta `timesteps`, `window` y parámetros del modelo según necesidad.
- Para visualizar en TensorBoard: `tensorboard --logdir logs_recurrentppo`